# Sentiment Analysis Assignment: AI-Assisted Build + Manual Debug

## Deliverable 1: Exact prompt given to the AI tool

> "Write a Python program that takes a list of food delivery customer reviews, preprocesses each review using tokenization, stopword removal, and stemming, builds a TF-IDF matrix, trains a Logistic Regression model to classify each review as Positive or Negative, prints the model's accuracy on a test split, and predicts the sentiment for three new reviews entered by the user at runtime."

## Deliverable 2a: AI's original code (run as-is)

In [1]:
"""
AI-GENERATED (ORIGINAL) VERSION
Prompt given to the AI tool:
"Write a Python program that takes a list of food delivery customer reviews,
preprocesses each review using tokenization, stopword removal, and stemming,
builds a TF-IDF matrix, trains a Logistic Regression model to classify each
review as Positive or Negative, prints the model's accuracy on a test split,
and predicts the sentiment for three new reviews entered by the user at runtime."
"""

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

# Hardcoded sample reviews + labels
reviews = [
    "The food was absolutely delicious and arrived hot",
    "Amazing service, will order again for sure",
    "The food was not good and arrived cold",
    "Terrible experience, the order was completely wrong",
    "I loved the taste, best delivery ever",
    "The delivery was fast and the packaging was great",
    "Worst food I have ever ordered, not fresh at all",
    "Not bad, but not great either",
    "Excellent quality and quick delivery",
    "The order never arrived, very disappointing",
]
labels = [
    "Positive", "Positive", "Negative", "Negative", "Positive",
    "Positive", "Negative", "Negative", "Positive", "Negative",
]

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess(text):
    tokens = word_tokenize(text.lower())
    tokens = [t for t in tokens if t.isalpha()]           # keep only words
    tokens = [t for t in tokens if t not in stop_words]   # remove stopwords
    tokens = [stemmer.stem(t) for t in tokens]            # stem
    return " ".join(tokens)

processed_reviews = [preprocess(r) for r in reviews]

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(processed_reviews)

X_train, X_test, y_train, y_test = train_test_split(
    X, labels, test_size=0.3, random_state=42
)

model = LogisticRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("Model accuracy on test split:", accuracy_score(y_test, y_pred))

# Predict on 3 new reviews
new_reviews = [
    "The food was not good",
    "Great taste and fast delivery",
    "Not fresh, very disappointing",
]

for r in new_reviews:
    processed = preprocess(r)
    vec = vectorizer.transform([processed])
    prediction = model.predict(vec)[0]
    print(f"Review: '{r}' -> Predicted: {prediction}  (preprocessed: '{processed}')")


Model accuracy on test split: 0.0
Review: 'The food was not good' -> Predicted: Negative  (preprocessed: 'food good')
Review: 'Great taste and fast delivery' -> Predicted: Negative  (preprocessed: 'great tast fast deliveri')
Review: 'Not fresh, very disappointing' -> Predicted: Negative  (preprocessed: 'fresh disappoint')


### Bug demonstration
The line below proves the bug directly: two **opposite-meaning** reviews collapse to the exact same preprocessed string, because stopword removal deletes the word 'not' along with genuinely meaningless words.

In [2]:
print("preprocess(\"The food was good\")     ->", repr(preprocess("The food was good")))
print("preprocess(\"The food was not good\") ->", repr(preprocess("The food was not good")))

preprocess("The food was good")     -> 'food good'
preprocess("The food was not good") -> 'food good'


## Deliverable 2b: Corrected version (fixed manually, without AI)

In [3]:
"""
CORRECTED VERSION (bug fixed manually, without AI help)

Bug found in the AI's original code:
Stopword removal deleted negation words ("not", "no", "never", etc.),
so "The food was not good" and "The food was good" both preprocessed
to the identical string "food good" -- the model could not distinguish
opposite-meaning reviews.

Fix: remove negation words from the stopword set before filtering, so
negation is preserved through tokenization and stemming.
"""

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

reviews = [
    "The food was absolutely delicious and arrived hot",
    "Amazing service, will order again for sure",
    "The food was not good and arrived cold",
    "Terrible experience, the order was completely wrong",
    "I loved the taste, best delivery ever",
    "The delivery was fast and the packaging was great",
    "Worst food I have ever ordered, not fresh at all",
    "Not bad, but not great either",
    "Excellent quality and quick delivery",
    "The order never arrived, very disappointing",
]
labels = [
    "Positive", "Positive", "Negative", "Negative", "Positive",
    "Positive", "Negative", "Negative", "Positive", "Negative",
]

# --- FIX: keep negation words out of the stopword removal set ---
negation_words = {"not", "no", "never", "n't", "none", "nobody", "nothing", "neither", "nor"}
stop_words = set(stopwords.words('english')) - negation_words

stemmer = PorterStemmer()

def preprocess(text):
    tokens = word_tokenize(text.lower())
    tokens = [t for t in tokens if t.isalpha()]
    tokens = [t for t in tokens if t not in stop_words]   # negation words survive now
    tokens = [stemmer.stem(t) for t in tokens]
    return " ".join(tokens)

processed_reviews = [preprocess(r) for r in reviews]

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(processed_reviews)

# --- Secondary fix: dataset is tiny, so use a smaller test_size (1 sample)
# with stratify to keep the split meaningful, and print train/test sizes
# so accuracy is read in context, not treated as a real benchmark number.
X_train, X_test, y_train, y_test = train_test_split(
    X, labels, test_size=0.2, random_state=42, stratify=labels
)
print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

model = LogisticRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("Model accuracy on test split:", accuracy_score(y_test, y_pred))

new_reviews = [
    "The food was not good",
    "Great taste and fast delivery",
    "Not fresh, very disappointing",
]

for r in new_reviews:
    processed = preprocess(r)
    vec = vectorizer.transform([processed])
    prediction = model.predict(vec)[0]
    print(f"Review: '{r}' -> Predicted: {prediction}  (preprocessed: '{processed}')")

print()
print("Sanity check -- negation now preserved:")
print("preprocess('The food was good')     ->", repr(preprocess('The food was good')))
print("preprocess('The food was not good') ->", repr(preprocess('The food was not good')))


Train size: 8, Test size: 2
Model accuracy on test split: 1.0
Review: 'The food was not good' -> Predicted: Negative  (preprocessed: 'food not good')
Review: 'Great taste and fast delivery' -> Predicted: Positive  (preprocessed: 'great tast fast deliveri')
Review: 'Not fresh, very disappointing' -> Predicted: Negative  (preprocessed: 'not fresh disappoint')

Sanity check -- negation now preserved:
preprocess('The food was good')     -> 'food good'
preprocess('The food was not good') -> 'food not good'


## Deliverable 3: What changed and why

The AI's stopword-removal step deleted negation words ('not', 'no', 'never', etc.) along with ordinary filler words, which caused opposite-meaning reviews like *"The food was good"* and *"The food was not good"* to preprocess to the identical string `food good` — the model literally couldn't tell them apart. I fixed this by removing negation words from the stopword set before filtering, so they survive tokenization and stemming intact. I also noticed the 30% test split left only 3 test samples from the 10-row dataset, making the accuracy score close to meaningless, so I reduced the test size and added stratification to keep the split representative.